In [13]:
import pandas as pd
import challenge_spotter.config as cfg

df= pd.read_csv(cfg.DATA_DIR / "train-test.csv")
print(len(df))
df.head(10)

48000


,load_id,pickup,delivery,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,equipment,weight,date,market_index,quote_signal,posted_rate
0,TR-000001,Richmond,Baltimore,38.09122,-76.78906,38.16908,-72.74564,274.3,Dry Van,30658.0,2025-01-01,0.95684,2.39595,645.41
1,TR-000002,Richmond,Philadelphia,38.09122,-76.78906,39.22317,-72.96710,280.5,Reefer,17555.0,2025-01-01,0.97623,2.43355,679.97
2,TR-000003,Philadelphia,Green Bay,39.22317,-72.96710,44.30296,-87.52871,967.8,Dry Van,31721.0,2025-01-01,1.00971,1.84491,1802.54
3,TR-000004,Hartford,Atlanta,39.55328,-72.18051,34.84933,-86.28940,965.4,Dry Van,32333.0,2025-01-01,0.94518,1.87712,1827.28
4,TR-000005,Dallas,Nashville,31.83025,-94.38343,35.29479,-88.08915,541.9,Reefer,35183.0,2025-01-01,0.98480,2.56300,1380.28
5,TR-000006,Phoenix,Corpus Christi,28.75668,-115.87725,28.61374,-95.89569,1396.4,Flatbed,17018.0,2025-01-01,0.95167,1.92983,2676.98
6,TR-000007,Fresno,San Antonio,33.07921,-119.79528,29.50969,-98.40059,1502.6,Dry Van,19112.0,2025-01-01,0.95696,1.82074,2718.54
7,TR-000008,San Francisco,Kansas City,35.19670,-121.69849,39.41104,-91.12450,1976.9,Dry Van,29169.0,2025-01-01,0.97818,1.78430,3522.92
8,TR-000009,Bakersfield,Montgomery,33.28832,-116.67935,31.37602,-86.10743,2027.7,Dry Van,35018.0,2025-01-01,0.96534,1.79522,3675.27
9,TR-000010,Albuquerque,Phoenix,35.39918,-109.59792,28.75668,-115.87725,717.1,Flatbed,34671.0,2025-01-01,0.98475,2.17079,1578.83


#my first concern is that the data may not be IID due temporal dependency. Therefore, in order to avoid data leakage, a temporal split will be implemented. Reserving the 20% of the most recent data for testing and the rest for training. 

In [14]:
#Seems like "delivery_lat" is not present in the print of describe().
df["delivery_lat"].dtype 

#The reason is an incorrect type. But casting directly to the same type as other geospatial features causes an error due to a corrupted value "28.6137   4"".
#so let's proceed droping the non numeric values before casting
df["delivery_lat"] = pd.to_numeric(df["delivery_lat"], errors="coerce").astype("float64")
df = df[df["delivery_lat"].notna()]
print(f"{len(df)} rows left")




47999 rows left


In [ ]:
#other columns are also persisted with wrong type
print(df.dtypes)

load_id         string[python]
pickup                category
delivery              category
pickup_lat             float64
pickup_lon             float64
delivery_lat           float64
delivery_lon           float64
distance               float64
equipment             category
weight                 float64
date            datetime64[ns]
market_index           float64
quote_signal           float64
posted_rate            float64
dtype: object


In [ ]:
#correcting data types.
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["pickup"] = df["pickup"].astype("category")
df["delivery"] = df["delivery"].astype("category")
df["equipment"] = df["equipment"].astype("category")
df["load_id"] = df["load_id"].astype("string")
print(df.dtypes)



load_id         string[python]
pickup                category
delivery              category
pickup_lat             float64
pickup_lon             float64
delivery_lat           float64
delivery_lon           float64
distance               float64
equipment             category
weight                 float64
date            datetime64[ns]
market_index           float64
quote_signal           float64
posted_rate            float64
dtype: object


In [58]:
df.describe()

,pickup_lat,pickup_lon,delivery_lat,delivery_lon,distance,weight,date,market_index,quote_signal,posted_rate
count,47999.000000,47999.000000,47999.000000,47999.000000,47999.000000,47699.000000,47999,47625.000000,47999.000000,47999.000000
mean,35.647517,-90.929088,35.641321,-90.857205,1135.859455,31028.620265,2025-05-31 19:58:00.897518336,1.083389,2.062468,2373.987868
min,28.357650,-121.698490,28.357650,-121.698490,70.000000,-47500.000000,2025-01-01 00:00:00,0.676390,0.692280,57.220000
25%,31.986910,-98.400590,31.986910,-98.400590,550.400000,25800.000000,2025-03-18 00:00:00,0.949670,1.891030,1251.550000
50%,35.294790,-88.089150,35.294790,-87.528710,953.300000,31436.000000,2025-05-31 00:00:00,1.055800,2.055750,2030.820000
75%,39.411040,-83.285060,39.411040,-83.285060,1645.550000,37018.000000,2025-08-15 00:00:00,1.219590,2.221690,3330.760000
max,44.302960,-69.500000,44.302960,-69.500000,3439.800000,47500.000000,2025-10-31 00:00:00,1.467780,3.610350,25533.000000
std,4.315326,13.482545,4.317125,13.476709,728.571747,9391.411936,NaN,0.168092,0.291394,1486.507896


In [ ]:
#checking missing values
df.isna().sum() 

load_id           0
pickup            0
delivery          0
pickup_lat        0
pickup_lon        0
delivery_lat      0
delivery_lon      0
distance          0
equipment         0
weight          300
date              0
market_index    374
quote_signal      0
posted_rate       0
dtype: int64


In [17]:
#we also have 292 invalid values in weight
print(f"invalid values in weight: {(df['weight'] < 0).sum()}")

#checking cardinality
print(f"cardinality in pickup {df['pickup'].nunique()}")
print(f"cardinality in delivery {df['delivery'].nunique()}")
print(f"cardinality in equipment {df['equipment'].nunique()}")


#show several rare categories (less than 1% even before the split, so i will check again after the split to consider agroup categories before OHE)
print(df["delivery"].value_counts())

invalid values in weight: 292
cardinality in pickup 64
cardinality in delivery 64
cardinality in equipment 3
delivery
Lexington      1197
Fort Wayne     1176
Baton Rouge    1167
Bakersfield    1156
Hartford       1143
               ... 
St. Louis       344
Birmingham      318
Detroit         307
Dallas          306
Las Vegas       292
Name: count, Length: 64, dtype: int64


In [25]:
#checking if the missing or invalid values are something structural.
invalid_weight= df["weight"] < 0
missing_weight= df["weight"].isna()
missing_market_index= df["market_index"].isna()
result1= ((invalid_weight) & (missing_market_index)).sum()
result2= ((missing_weight) & (missing_market_index)).sum()
print(f"rows with invalid weight and market_index as missing value {result1}")
print(f"rows with weight and market index as missing values {result2}")
#i decide to proceed in the next way:
#1- inputing nan in the negative values of weight and treat everything as the same set. 
#2- Droping the NaN in "market_index".

rows with invalid weight and market_index as missing value 3
rows with weight and market index as missing values 1
